# LiTS Next Phase: Pairing Audit and Overfit Sanity Gate

This notebook is the controlled next step after discovering that the previous 512×512 image directory did not spatially match the 256×256 tumor masks.

It does four things:

1. Builds a filename-level index for the likely corrected 256×256 image source and the existing masks.
2. Produces side-by-side legacy-versus-corrected overlays for manual alignment review.
3. Tracks project progress and saves audit evidence in `Practice/next_phase_outputs/`.
4. Runs a small canonical-model overfit test only after you manually approve the overlays.

It never edits the source images or masks and never accesses the held-out test set for model selection.

## How to run

1. Open this notebook from the project root or from `Practice/`.
2. Select the project `.venv` Python kernel.
3. Run cells from the top through the alignment comparison.
4. Inspect every corrected overlay carefully.
5. In the approval cell, change `VISUAL_ALIGNMENT_APPROVED = True` only if corrected contours are anatomically plausible.
6. Change `RUN_OVERFIT_SANITY = True`, then run the remaining cells.
7. Share `alignment_audit.json`, `next_phase_summary.json`, `overfit_history.csv` (if created), and the saved PNGs with Codex.

Expected first-pass audit time: several minutes. Expected overfit test: roughly 5–15 minutes on the RTX 3050 Ti.

In [ ]:
from pathlib import Path
import sys, re, json, math, time, random
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from IPython.display import display

import torch
from torch.utils.data import Dataset, DataLoader

# Locate the project whether the notebook starts in Liver/ or Liver/Practice/.
cwd = Path.cwd().resolve()
if (cwd / "Practice").is_dir():
    PROJECT_ROOT = cwd
elif cwd.name.lower() == "practice" and (cwd.parent / "src").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    candidates = [p for p in [cwd, *cwd.parents] if (p / "Practice").is_dir() and (p / "src").is_dir()]
    if not candidates:
        raise RuntimeError("Could not locate the Liver project root")
    PROJECT_ROOT = candidates[0]

PRACTICE_DIR = PROJECT_ROOT / "Practice"
OUTPUT_DIR = PRACTICE_DIR / "next_phase_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\Dataset")
CORRECT_IMAGES_DIR = DATASET_ROOT / "lits-png" / "dataset_6" / "dataset_6"
LEGACY_IMAGES_DIR = DATASET_ROOT / "Liver Img Dataset"
MASKS_DIR = DATASET_ROOT / "LiTS_masks"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project: {PROJECT_ROOT}")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Device:  {DEVICE}")
print(f"Corrected image source exists: {CORRECT_IMAGES_DIR.is_dir()}")
print(f"Legacy image source exists:    {LEGACY_IMAGES_DIR.is_dir()}")
print(f"Mask source exists:            {MASKS_DIR.is_dir()}")

## 1. Build and verify the three indexes

The corrected candidate uses `volume-{volume}_{slice}.png`. The previous invalid source used `Volume-{volume}-{slice}.png`. Masks use `mask-{volume}-{slice}.png`.

The audit requires identical corrected-image and mask key sets; file counts alone are not sufficient.

In [ ]:
SOURCE_RE = re.compile(r"volume-(\d+)_(\d+)\.png$", re.IGNORECASE)
LEGACY_RE = re.compile(r"volume-(\d+)-(\d+)\.png$", re.IGNORECASE)
MASK_RE = re.compile(r"mask-(\d+)-(\d+)\.png$", re.IGNORECASE)

def build_index(folder, pattern, glob_pattern):
    index = {}
    duplicates = []
    for path in tqdm(list(folder.glob(glob_pattern)), desc=f"Indexing {folder.name}"):
        match = pattern.match(path.name)
        if not match:
            continue
        key = (int(match.group(1)), int(match.group(2)))
        if key in index:
            duplicates.append(key)
        index[key] = path
    return index, duplicates

source_index, source_duplicates = build_index(CORRECT_IMAGES_DIR, SOURCE_RE, "volume-*_*.png")
legacy_index, legacy_duplicates = build_index(LEGACY_IMAGES_DIR, LEGACY_RE, "Volume-*-*.png")
mask_index, mask_duplicates = build_index(MASKS_DIR, MASK_RE, "mask-*-*.png")

source_keys = set(source_index)
mask_keys = set(mask_index)
legacy_keys = set(legacy_index)
missing_masks = sorted(source_keys - mask_keys)
missing_corrected_images = sorted(mask_keys - source_keys)

sample_keys = sorted(source_keys & mask_keys)
dimension_sample = sample_keys[::max(1, len(sample_keys) // 256)][:256]
dimension_mismatches = []
for key in tqdm(dimension_sample, desc="Checking sampled dimensions"):
    with Image.open(source_index[key]) as image, Image.open(mask_index[key]) as mask:
        if image.size != mask.size:
            dimension_mismatches.append({"key": key, "image_size": image.size, "mask_size": mask.size})

audit = {
    "corrected_images": len(source_index),
    "legacy_images": len(legacy_index),
    "masks": len(mask_index),
    "matched_corrected_pairs": len(source_keys & mask_keys),
    "source_duplicates": len(source_duplicates),
    "mask_duplicates": len(mask_duplicates),
    "missing_masks": len(missing_masks),
    "missing_corrected_images": len(missing_corrected_images),
    "dimension_sample_count": len(dimension_sample),
    "dimension_mismatches": len(dimension_mismatches),
    "expected_pair_count": 58638,
}
audit["pair_gate_passed"] = bool(
    audit["matched_corrected_pairs"] == audit["expected_pair_count"]
    and not source_duplicates and not mask_duplicates
    and not missing_masks and not missing_corrected_images
    and not dimension_mismatches
)

(OUTPUT_DIR / "alignment_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
display(pd.DataFrame([audit]).T.rename(columns={0: "value"}))
print("PAIR GATE:", "PASS" if audit["pair_gate_passed"] else "FAIL")

## 2. Build the cached mask manifest

The first run scans every mask and may take several minutes. Later runs reuse `mask_manifest.csv`.

This provides tumor-positive slice counts, pixel burden, split membership, and representative slices for visual review.

In [ ]:
manifest_path = OUTPUT_DIR / "mask_manifest.csv"

def load_splits():
    split_map = {}
    for split in ["train", "val", "test"]:
        path = SPLITS_DIR / f"{split}_volumes.txt"
        if path.exists():
            for line in path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line:
                    split_map[int(line)] = split
    return split_map

split_map = load_splits()

if manifest_path.exists():
    mask_manifest = pd.read_csv(manifest_path)
    if len(mask_manifest) != len(mask_index):
        print("Cached manifest count changed; rebuilding.")
        mask_manifest = None
else:
    mask_manifest = None

if mask_manifest is None:
    rows = []
    for (volume_id, slice_id), path in tqdm(sorted(mask_index.items()), desc="Scanning masks"):
        with Image.open(path) as image:
            mask = np.asarray(image.convert("L"))
        tumor_pixels = int(np.count_nonzero(mask))
        rows.append({
            "volume_id": volume_id,
            "slice_id": slice_id,
            "split": split_map.get(volume_id, "unknown"),
            "tumor_pixels": tumor_pixels,
            "has_tumor": tumor_pixels > 0,
            "mask_height": int(mask.shape[0]),
            "mask_width": int(mask.shape[1]),
        })
    mask_manifest = pd.DataFrame(rows)
    mask_manifest.to_csv(manifest_path, index=False)

mask_manifest["has_tumor"] = mask_manifest["has_tumor"].map(
    lambda value: value if isinstance(value, (bool, np.bool_)) else str(value).strip().lower() == "true"
)
positive_manifest = mask_manifest[mask_manifest["has_tumor"]].copy()
volume_manifest = mask_manifest.groupby(["volume_id", "split"], as_index=False).agg(
    slices=("slice_id", "count"),
    positive_slices=("has_tumor", "sum"),
    tumor_pixels=("tumor_pixels", "sum"),
)
volume_manifest["positive_slice_pct"] = 100 * volume_manifest["positive_slices"] / volume_manifest["slices"]
volume_manifest.to_csv(OUTPUT_DIR / "volume_manifest.csv", index=False)

split_summary = mask_manifest.groupby("split", as_index=False).agg(
    slices=("slice_id", "count"),
    positive_slices=("has_tumor", "sum"),
    tumor_pixels=("tumor_pixels", "sum"),
)
split_summary["positive_slice_pct"] = 100 * split_summary["positive_slices"] / split_summary["slices"]
display(split_summary)
print(f"Tumor-positive masks: {len(positive_manifest):,} / {len(mask_manifest):,}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

ordered = split_summary.set_index("split").reindex(["train", "val", "test"]).reset_index()
axes[0].bar(ordered["split"], ordered["positive_slice_pct"], color=["#4c78a8", "#f2cf5b", "#e45756"])
axes[0].set_title("Tumor-positive slices by split")
axes[0].set_ylabel("Positive slices (%)")
for i, value in enumerate(ordered["positive_slice_pct"]):
    axes[0].text(i, value, f"{value:.1f}%", ha="center", va="bottom")

positive_pixels = positive_manifest["tumor_pixels"].to_numpy()
axes[1].hist(positive_pixels, bins=50, color="#7a5195", alpha=0.85)
axes[1].set_xscale("log")
axes[1].set_title("Positive mask-size distribution")
axes[1].set_xlabel("Tumor pixels per slice (log scale)")
axes[1].set_ylabel("Slices")

for split, group in volume_manifest.groupby("split"):
    axes[2].scatter(group["slices"], group["positive_slices"], label=split, alpha=0.75)
axes[2].set_title("Tumor-positive slices per volume")
axes[2].set_xlabel("Total slices")
axes[2].set_ylabel("Positive slices")
axes[2].legend()

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "dataset_distribution.png", dpi=170, bbox_inches="tight")
plt.show()

## 3. Visual pairing comparison

Each row shows the same mask against:

- the previous 512×512 image after resizing to 256×256;
- the corrected 256×256 LiTS PNG source;
- the mask alone.

A corrected green contour must lie on a plausible lesion inside abdominal anatomy. Do not approve the audit if contours remain on scanner rails, empty space, or unrelated organs.

In [ ]:
representatives = (
    positive_manifest[positive_manifest["split"].isin(["train", "val"])]
    .sort_values("tumor_pixels", ascending=False)
    .drop_duplicates("volume_id")
    .head(8)
)
representative_keys = [
    (int(row.volume_id), int(row.slice_id))
    for row in representatives.itertuples()
]

def load_gray(path, size=(256, 256)):
    with Image.open(path) as image:
        image = image.convert("L")
        if image.size != size:
            image = image.resize(size, Image.Resampling.BILINEAR)
        return np.asarray(image)

fig, axes = plt.subplots(len(representative_keys), 3, figsize=(13, 4 * len(representative_keys)))
if len(representative_keys) == 1:
    axes = np.array([axes])

for row, key in enumerate(representative_keys):
    corrected = load_gray(source_index[key])
    legacy = load_gray(legacy_index[key])
    with Image.open(mask_index[key]) as image:
        mask = np.asarray(image.convert("L")) > 0

    for column, (image, title) in enumerate([
        (legacy, "Previous 512→256 pairing"),
        (corrected, "Candidate corrected pairing"),
    ]):
        axes[row, column].imshow(image, cmap="gray")
        axes[row, column].contour(mask, levels=[0.5], colors=["lime"], linewidths=1.2)
        axes[row, column].set_title(f"{title}\nvolume {key[0]}, slice {key[1]}")
        axes[row, column].axis("off")

    axes[row, 2].imshow(mask, cmap="gray")
    axes[row, 2].set_title(f"Tumor mask ({int(mask.sum()):,} px)")
    axes[row, 2].axis("off")

fig.tight_layout()
comparison_path = OUTPUT_DIR / "legacy_vs_corrected_alignment.png"
fig.savefig(comparison_path, dpi=170, bbox_inches="tight")
plt.show()
print(f"Saved: {comparison_path}")

## 4. Manual approval and run controls

After reviewing every corrected overlay above:

- change `VISUAL_ALIGNMENT_APPROVED` to `True` only if the corrected contours are plausible;
- change `RUN_OVERFIT_SANITY` to `True` to run the canonical model on 16 large positive masks.

The overfit test is deliberately blocked until manual alignment approval.

In [ ]:
# MANUAL GATE — edit these two values after inspecting the overlays.
VISUAL_ALIGNMENT_APPROVED = False
RUN_OVERFIT_SANITY = False

OVERFIT_SLICES = 16
OVERFIT_MAX_EPOCHS = 120
OVERFIT_TARGET_DICE = 0.80
OVERFIT_PRETRAINED = True

print("Visual alignment:", "APPROVED" if VISUAL_ALIGNMENT_APPROVED else "NOT YET APPROVED")
print("Overfit sanity:", "ENABLED" if RUN_OVERFIT_SANITY else "DISABLED")

In [ ]:
def load_existing_overfit_result():
    path = OUTPUT_DIR / "overfit_result.json"
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {"status": "not_run", "best_dice": None, "gate_passed": False}

def render_progress(overfit_result=None):
    if overfit_result is None:
        overfit_result = load_existing_overfit_result()
    stages = [
        ("Runtime and paths", all([CORRECT_IMAGES_DIR.is_dir(), MASKS_DIR.is_dir(), DEVICE is not None])),
        ("58,638 corrected pairs", audit["pair_gate_passed"]),
        ("Visual alignment approval", bool(VISUAL_ALIGNMENT_APPROVED)),
        ("Overfit Dice ≥ 0.80", bool(overfit_result.get("gate_passed", False))),
        ("Corrected 5-epoch baseline", False),
        ("FAUP/UWACL comparison", False),
        ("Held-out test evaluation", False),
    ]
    completed = sum(int(done) for _, done in stages)
    progress_pct = 100 * completed / len(stages)

    fig, ax = plt.subplots(figsize=(11, 5))
    y = np.arange(len(stages))
    colors = ["#2ca02c" if done else "#c7c7c7" for _, done in stages]
    ax.barh(y, [1] * len(stages), color=colors)
    ax.set_yticks(y, [name for name, _ in stages])
    ax.set_xlim(0, 1)
    ax.set_xticks([])
    ax.invert_yaxis()
    for index, (_, done) in enumerate(stages):
        ax.text(0.5, index, "COMPLETE" if done else "PENDING", ha="center", va="center",
                color="white" if done else "black", fontweight="bold")
    ax.set_title(f"Research validation progress: {progress_pct:.0f}% ({completed}/{len(stages)} gates)")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "phase_progress.png", dpi=170, bbox_inches="tight")
    plt.show()

    progress = {
        "completed_stages": completed,
        "total_stages": len(stages),
        "progress_pct": progress_pct,
        "stages": [{"name": name, "complete": bool(done)} for name, done in stages],
    }
    (OUTPUT_DIR / "progress.json").write_text(json.dumps(progress, indent=2), encoding="utf-8")
    return progress

progress = render_progress()
display(pd.DataFrame(progress["stages"]))

## 5. Canonical-model overfit sanity test

This test uses the corrected candidate images, the existing masks, the real MobileNetV2-U-Net, and the real Focal-Dice loss. It selects the 16 largest positive masks from training volumes only.

Success criterion: best micro Dice ≥ 0.80. Failure means the data/loss/model path must be debugged before another full baseline.

In [ ]:
from src.framework.models.mobilenetv2_unet import MobileNetV2UNet
from src.framework.losses.focal_dice import FocalDiceLoss

class CachedSliceDataset(Dataset):
    def __init__(self, keys):
        images, masks = [], []
        for key in keys:
            image = load_gray(source_index[key]).astype(np.float32) / 255.0
            with Image.open(mask_index[key]) as mask_image:
                mask = (np.asarray(mask_image.convert("L")) > 0).astype(np.float32)
            images.append(torch.from_numpy(image).unsqueeze(0))
            masks.append(torch.from_numpy(mask).unsqueeze(0))
        self.images = torch.stack(images)
        self.masks = torch.stack(masks)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        return self.images[index], self.masks[index]

def micro_dice(pred, target):
    intersection = (pred * target).sum().item()
    denominator = pred.sum().item() + target.sum().item()
    return 1.0 if denominator == 0 else 2.0 * intersection / denominator

def run_overfit_sanity():
    if not RUN_OVERFIT_SANITY:
        print("Overfit test is disabled. Approve the overlays, set RUN_OVERFIT_SANITY=True, and rerun this cell.")
        return {"status": "not_run", "best_dice": None, "gate_passed": False}
    if not VISUAL_ALIGNMENT_APPROVED:
        raise RuntimeError("Set VISUAL_ALIGNMENT_APPROVED=True only after reviewing corrected overlays.")

    train_positive = positive_manifest[
        positive_manifest["volume_id"].isin(range(104))
    ].sort_values("tumor_pixels", ascending=False).head(OVERFIT_SLICES)
    keys = [(int(row.volume_id), int(row.slice_id)) for row in train_positive.itertuples()]
    dataset = CachedSliceDataset(keys)
    loader = DataLoader(
        dataset, batch_size=4, shuffle=True,
        generator=torch.Generator().manual_seed(SEED), num_workers=0,
    )

    model = MobileNetV2UNet(
        in_channels=1, out_channels=1, pretrained=OVERFIT_PRETRAINED
    ).to(DEVICE)
    criterion = FocalDiceLoss(
        focal_alpha=0.75, focal_gamma=2.0, focal_weight=0.5, dice_weight=0.5
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    thresholds = np.arange(0.10, 0.91, 0.05)

    history = []
    best = {"dice": -1.0, "threshold": 0.5, "epoch": 0}
    consecutive_passes = 0
    started = time.perf_counter()

    for epoch in range(1, OVERFIT_MAX_EPOCHS + 1):
        model.train()
        losses = []
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        model.eval()
        probabilities, targets = [], []
        with torch.no_grad():
            for images, masks in DataLoader(dataset, batch_size=4, shuffle=False):
                probabilities.append(torch.sigmoid(model(images.to(DEVICE))).cpu())
                targets.append(masks)
        probabilities = torch.cat(probabilities)
        targets = torch.cat(targets)

        threshold_scores = []
        for threshold in thresholds:
            score = micro_dice((probabilities >= threshold).float(), targets)
            threshold_scores.append((score, float(threshold)))
        epoch_dice, epoch_threshold = max(threshold_scores)

        history.append({
            "epoch": epoch,
            "loss": float(np.mean(losses)),
            "best_dice": epoch_dice,
            "best_threshold": epoch_threshold,
        })
        if epoch_dice > best["dice"]:
            best = {"dice": epoch_dice, "threshold": epoch_threshold, "epoch": epoch}

        consecutive_passes = consecutive_passes + 1 if epoch_dice >= OVERFIT_TARGET_DICE else 0
        if epoch == 1 or epoch % 10 == 0 or consecutive_passes:
            print(
                f"epoch={epoch:03d} loss={np.mean(losses):.5f} "
                f"dice={epoch_dice:.4f} threshold={epoch_threshold:.2f}"
            )
        if consecutive_passes >= 3:
            print("Overfit gate held for three consecutive epochs.")
            break

    history_df = pd.DataFrame(history)
    history_df.to_csv(OUTPUT_DIR / "overfit_history.csv", index=False)

    result = {
        "status": "completed",
        "model": "MobileNetV2UNet",
        "slices": len(dataset),
        "best_dice": float(best["dice"]),
        "best_threshold": float(best["threshold"]),
        "best_epoch": int(best["epoch"]),
        "epochs_run": len(history),
        "runtime_seconds": time.perf_counter() - started,
        "gate_target": OVERFIT_TARGET_DICE,
        "gate_passed": bool(best["dice"] >= OVERFIT_TARGET_DICE),
        "keys": [[int(v), int(s)] for v, s in keys],
    }
    (OUTPUT_DIR / "overfit_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(history_df["epoch"], history_df["loss"], color="#4c78a8")
    axes[0].set_title("Overfit training loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Focal-Dice loss")
    axes[1].plot(history_df["epoch"], history_df["best_dice"], color="#e45756")
    axes[1].axhline(OVERFIT_TARGET_DICE, color="black", linestyle="--", label="Gate: 0.80")
    axes[1].set_ylim(0, 1)
    axes[1].set_title("Best thresholded micro Dice")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Dice")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "overfit_curves.png", dpi=170, bbox_inches="tight")
    plt.show()

    model.eval()
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    with torch.no_grad():
        probs = torch.sigmoid(model(dataset.images[:4].to(DEVICE))).cpu()
    preds = probs >= best["threshold"]
    for index in range(4):
        image = dataset.images[index, 0].numpy()
        truth = dataset.masks[index, 0].numpy() > 0
        pred = preds[index, 0].numpy()
        axes[0, index].imshow(image, cmap="gray")
        axes[0, index].contour(truth, levels=[0.5], colors=["lime"])
        axes[0, index].set_title(f"Ground truth {index + 1}")
        axes[1, index].imshow(image, cmap="gray")
        axes[1, index].contour(pred, levels=[0.5], colors=["red"])
        axes[1, index].set_title(f"Prediction {index + 1}")
        axes[0, index].axis("off")
        axes[1, index].axis("off")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "overfit_predictions.png", dpi=170, bbox_inches="tight")
    plt.show()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

OVERFIT_RESULT = run_overfit_sanity()
display(pd.DataFrame([OVERFIT_RESULT]).T.rename(columns={0: "value"}))

## 6. Final gate and saved progress summary

A corrected baseline is not authorized by this notebook alone. When the pair gate, manual visual gate, and overfit gate all pass, share the saved evidence with Codex so the framework loader and experiment configuration can be switched safely to the corrected filename layout.

In [ ]:
overfit_result = (
    OVERFIT_RESULT if "OVERFIT_RESULT" in globals()
    else load_existing_overfit_result()
)
progress = render_progress(overfit_result)

ready_for_framework_repair = bool(
    audit["pair_gate_passed"]
    and VISUAL_ALIGNMENT_APPROVED
    and overfit_result.get("gate_passed", False)
)

summary = {
    "pair_gate_passed": bool(audit["pair_gate_passed"]),
    "visual_alignment_approved": bool(VISUAL_ALIGNMENT_APPROVED),
    "overfit_status": overfit_result.get("status"),
    "overfit_best_dice": overfit_result.get("best_dice"),
    "overfit_gate_passed": bool(overfit_result.get("gate_passed", False)),
    "progress_pct": progress["progress_pct"],
    "ready_for_framework_repair": ready_for_framework_repair,
    "previous_baseline_valid": False,
    "previous_baseline_reason": "512x512 image source was spatially mismatched with 256x256 tumor masks",
}
(OUTPUT_DIR / "next_phase_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

lines = [
    "LITS NEXT-PHASE SUMMARY",
    "=" * 72,
    f"Pair gate:             {'PASS' if summary['pair_gate_passed'] else 'FAIL'}",
    f"Visual approval:       {'PASS' if summary['visual_alignment_approved'] else 'PENDING'}",
    f"Overfit gate:          {'PASS' if summary['overfit_gate_passed'] else 'PENDING/FAIL'}",
    f"Best overfit Dice:     {summary['overfit_best_dice']}",
    f"Overall progress:      {summary['progress_pct']:.1f}%",
    f"Ready for code repair: {summary['ready_for_framework_repair']}",
    "",
    "Do not run FAUP/UWACL or the held-out test set from this notebook.",
]
(OUTPUT_DIR / "next_phase_summary.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))
if ready_for_framework_repair:
    print("All audit gates passed. Share the output files with Codex for framework-loader repair.")
else:
    print("One or more gates are pending or failed. Do not restart full baseline training yet.")

## Files to share after running

From `Practice/next_phase_outputs/`:

- `alignment_audit.json`
- `legacy_vs_corrected_alignment.png`
- `dataset_distribution.png`
- `next_phase_summary.json`
- `phase_progress.png`
- `overfit_history.csv`, `overfit_result.json`, `overfit_curves.png`, and `overfit_predictions.png` if the overfit test ran

Do not share checkpoints or the original dataset files.